### Perf (Curves and Grid): 8020-intact animals

In [ ]:
import numpy as np
import seaborn as sns
from matplotlib.colors import Normalize
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    TIER_TITLES,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)

# Rebuild if stale:
# from mab_builders.build_all import build
# build("perf_tier_expert", "abstract_perf_probability_matrix", n_jobs=8)
gd = mab_subjects.GroupData()

# learning curves: intact sessions trimmed from the expertise day
curve_df = select(gd.perf_tier_expert.latest, paradigm="8020", lesion="intact")
curve_df = curve_df[curve_df["n_sessions"] >= 10]  # min sessions per tier per animal

# prob grids: mean P(High) over the last 90 trials of each combination (Cosyne product)
grid_df = select(
    gd.abstract_perf_probability_matrix.latest, paradigm="8020", lesion="intact"
)
probs, means, counts = prob_grid_group_means(grid_df, "perf_mat")
means["diff"] = means["struc"] - means["unstruc"]  # NaN unless both groups saw the pair

# keep only probability levels either group saw
seen = np.zeros(len(probs), dtype=bool)
for g in ["unstruc", "struc"]:
    seen |= np.isfinite(means[g]).any(axis=0) | np.isfinite(means[g]).any(axis=1)
probs = probs[seen]
means = {k: m[np.ix_(seen, seen)] for k, m in means.items()}

palette = Palette2Arm().as_dict()
hue_order = ["unstruc", "struc"]
bottom_tiers = ["low-low", "high-low", "high-high", "dependent", "independent"]

fig = plotting.Fig(2, 5, size=(18, 8), fontsize=14, axis_lw=1.5, tick_size=5)


def plot_curve(ax, tier, ylabel):
    sns.lineplot(
        data=curve_df[curve_df["tier_label"] == tier],
        x="trial_id",
        y="tier_perf",
        hue="group",
        hue_order=hue_order,
        palette=palette,
        lw=2.5,
        err_kws=dict(edgecolor="none", alpha=0.3),
        errorbar="se",
        legend=False,
        ax=ax,
    )
    ax.set_ylim(0.4, 1)
    ax.set_yticks([0.4, 0.6, 0.8, 1.0])  # first and last ticks sit on the axis limits
    ax.grid(axis="y", zorder=-1, alpha=0.5)
    ax.set_ylabel("P(High)" if ylabel else "")
    ax.set_xlabel("Trials")
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_title(TIER_TITLES[tier])


# ---- top row: overall curve + prob grids (unstruc, struc, S - U) ----
plot_curve(fig.subplot(fig.gs[0, 0]), "all", ylabel=True)

grid_gs = fig.gs[0, 1:].subgridspec(1, 3)
group_norm = Normalize(  # shared scale so unstruc vs struc is comparable
    vmin=np.floor(np.nanmin([np.nanmin(means[g]) for g in hue_order]) * 20) / 20,
    vmax=np.ceil(np.nanmax([np.nanmax(means[g]) for g in hue_order]) * 20) / 20,
)
grid_axes = []
for i, g in enumerate(hue_order):
    ax = fig.subplot(grid_gs[i])
    im_group = draw_prob_grid(
        ax, means[g], probs, has_diag=False, cmap="mako", norm=group_norm
    )
    ax.set_title(f"{g} (n={counts[g]})", color=palette[g], fontweight="bold")
    if i > 0:
        ax.set_ylabel("")
    grid_axes.append(ax)
fig.fig.colorbar(im_group, ax=grid_axes, label="P(High)", shrink=0.8)

ax = fig.subplot(grid_gs[2])
im_diff = draw_prob_grid(
    ax,
    means["diff"],
    probs,
    has_diag=False,  # P(High) is undefined for equal probabilities
    **diverging_bins(np.nanmax(np.abs(means["diff"])), step=BIN_STEPS["perf_mat"]),
)
ax.set_title("struc − unstruc")
ax.set_ylabel("")
fig.fig.colorbar(im_diff, ax=ax, label="Diff. P(High)", shrink=0.8)

# ---- bottom row: learning curves per probability tier ----
for i, tier in enumerate(bottom_tiers):
    plot_curve(fig.subplot(fig.gs[1, i]), tier, ylabel=i == 0)

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "perf_8020_intact", format="png", dpi=300)
fig.savefig(figdir / "perf_8020_intact", format="svg")

In [ ]:
import mab_subjects
from mab_builders.build_all import build

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
)
# abstract_swp_around_flip already exists from the Cosyne notebook; add it here if stale
build("swp_by_block_expert", exps=exps, n_jobs=6)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)
from statplotannot.plots import SeabornPlotter

gd = mab_subjects.GroupData()
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]

# P(Switch) per trial: all blocks, first and second block of each window
block_df = select(gd.swp_by_block_expert.latest, paradigm="8020", lesion="intact")

# P(Switch) from the previous block's last 10 trials into the new block's first 50,
# split by whether the better port flipped or stayed on the same port
flip_df = select(gd.abstract_swp_around_flip.latest, paradigm="8020", lesion="intact")

# prob grid: P(Switch) averaged over trials 2-100 of each combination (Cosyne product)
grid_df = select(
    gd.abstract_swp_probability_matrix.latest, paradigm="8020", lesion="intact"
)
probs, grid_means, grid_counts = prob_grid_group_means(grid_df, "swp_mat")
swp_diff = grid_means["struc"] - grid_means["unstruc"]  # NaN unless both saw the pair
seen = np.isfinite(swp_diff).any(axis=0) | np.isfinite(swp_diff).any(axis=1)
probs, swp_diff = probs[seen], swp_diff[np.ix_(seen, seen)]

# each animal's peak flip response after the transition, as in the Cosyne abstract:
# P(Switch | flipped) - P(Switch | same), gaussian-smoothed; its trial and height
sigma = 2  # smoothing (trials)
rows = []
for (group, name), d in flip_df[flip_df["trial_id"] > 0].groupby(["group", "name"]):
    wide = d.pivot_table(
        index="trial_id", columns="prev_best_arm", values="switch_prob"
    )
    if not {"flipped", "same"} <= set(wide.columns):
        continue
    smooth = gaussian_filter1d(
        np.nan_to_num((wide["flipped"] - wide["same"]).to_numpy()), sigma
    )
    rows.append(
        dict(
            kind="Animals",
            group=group,
            name=name,
            peak_trial=wide.index.to_numpy()[np.argmax(smooth)],
            peak_swp=smooth.max(),
        )
    )
peaks = pd.DataFrame(rows)

fig = plotting.Fig(2, 4, size=(17, 8), fontsize=14, axis_lw=1.5, tick_size=5)


def plot_swp(ax, data, title):
    sns.lineplot(
        data=data,
        x="trial_id",
        y="switch_prob",
        hue="group",
        hue_order=hue_order,
        palette=palette,
        lw=2.5,
        err_kws=dict(edgecolor="none", alpha=0.3),
        errorbar="se",
        legend=False,
        ax=ax,
    )
    ax.grid(axis="y", zorder=-1, alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel("")


# ---- top row: P(Switch) over all blocks, block 1, block 2 (same width as bottom panels) ----
top_axes = []
for i, (label, title) in enumerate(
    [("all", "All blocks"), ("block1", "Block 1"), ("block2", "Block 2")]
):
    ax = fig.subplot(fig.gs[0, i])
    plot_swp(ax, block_df[block_df["block_label"] == label], title)
    ax.set_xlim(1, 100)
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_xlabel("Trials")
    top_axes.append(ax)

ax = fig.subplot(fig.gs[0, 3])
im = draw_prob_grid(
    ax,
    swp_diff,
    probs,
    has_diag=np.isfinite(
        np.diag(swp_diff)
    ).any(),  # switching is defined on the diagonal
    **diverging_bins(np.nanmax(np.abs(swp_diff)), step=BIN_STEPS["swp_mat"]),
)
ax.set_title(f"struc − unstruc (n={grid_counts['struc']} vs {grid_counts['unstruc']})")
fig.fig.colorbar(im, ax=ax, label="Diff. P(Switch)", shrink=0.8)

# ---- bottom row: around the block transition, flipped / same ----
bottom_axes = []
for i, (data, title) in enumerate(
    [
        (flip_df[flip_df["prev_best_arm"] == "flipped"], "Better port flipped"),
        (flip_df[flip_df["prev_best_arm"] == "same"], "Better port same"),
    ]
):
    ax = fig.subplot(fig.gs[1, i])
    plot_swp(ax, data, title)
    ax.axvline(0, color="gray", ls=":", lw=1.5)  # block transition
    ax.set_xlim(-10, 50)
    ax.set_xticks([-10, 0, 10, 20, 30, 40, 50])
    ax.set_xlabel("Trials from transition")
    bottom_axes.append(ax)

# shared y-scale within each row; bottom and top ticks sit on the axis limits
for axes in [top_axes, bottom_axes]:
    hi = max(ax.dataLim.y1 for ax in axes)
    for ax in axes:
        three_yticks(ax, lo=0.0, hi=hi)
top_axes[0].set_ylabel("P(Switch)")
bottom_axes[0].set_ylabel("P(Switch)")


# ---- flip response peak per animal, trial and height: boxplot + Mann-Whitney ----
def plot_peak_box(ax, y, ylabel, set_yaxis):
    plotter = (
        SeabornPlotter(
            data=peaks,
            x="kind",
            y=y,
            hue="group",
            hue_order=hue_order,
            ax=ax,
        )
        .boxplot_filled(palette=palette, sep=0.4)
        .stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=7,
            jitter=0.03,
            offset=0.1,  # dots beside, not on, their box
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
    )
    set_yaxis(ax)  # before the brackets (data units)
    plotter.stat_classical(
        "Mann-Whitney",
        pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
        loc="outside",
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlabel("")
    ax.set_xticks([])
    ax.set_ylabel(ylabel)


plot_peak_box(
    fig.subplot(fig.gs[1, 2]),
    "peak_swp",
    "Flip response\npeak Diff. P(Switch)",
    lambda ax: three_yticks(ax, lo=0.0),
)

plot_peak_box(
    fig.subplot(fig.gs[1, 3]),
    "peak_trial",
    "Flip response\npeak (trial)",
    lambda ax: (ax.set_ylim(0, 50), ax.set_yticks([0, 25, 50])),
)
n = peaks.groupby("group")["name"].nunique()
print(n.to_dict())
print(peaks.groupby("group")[["peak_trial", "peak_swp"]].median().to_dict())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "swp_8020_intact", format="png", dpi=300)
fig.savefig(figdir / "swp_8020_intact", format="svg")

In [ ]:
import mab_subjects
from mab_builders.build_all import build

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
    + mab_subjects.unstruc_rnn.p8020_good_sess
    + mab_subjects.struc_rnn.p8020_good_sess
)
# the prob-grid _rnn products already exist from the Cosyne notebook; add them if stale
build("abstract_swp_around_flip_rnn", exps=exps, n_jobs=6)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)
from statplotannot.plots import SeabornPlotter

gd = mab_subjects.GroupData()
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]
kinds = {"animal": "Animals", "rnn": "RNNs"}


def animal_and_rnn(df):
    """8020 intact animals and 8020 RNN models, tagged by 'kind'."""
    return pd.concat(
        [
            select(df, paradigm="8020", lesion="intact").assign(kind="animal"),
            select(df, paradigm="8020", rnn=True).assign(kind="rnn"),
        ]
    )


def s_minus_u_grids(df, mat_col):
    """struc - unstruc grid per kind, on probability axes shared by both kinds."""
    df = df.assign(group=df["group"] + "_" + df["kind"])
    probs, means, counts = prob_grid_group_means(
        df, mat_col, groups=[f"{g}_{k}" for k in kinds for g in hue_order]
    )
    diffs = {k: means[f"struc_{k}"] - means[f"unstruc_{k}"] for k in kinds}
    seen = np.zeros(len(probs), dtype=bool)
    for d in diffs.values():
        seen |= np.isfinite(d).any(axis=0) | np.isfinite(d).any(axis=1)
    diffs = {k: d[np.ix_(seen, seen)] for k, d in diffs.items()}
    return probs[seen], diffs, counts


# prob grids (Cosyne products): P(High) over the last 90 trials, P(Switch) over trials 2-100
perf_grids = s_minus_u_grids(
    animal_and_rnn(gd.abstract_perf_probability_matrix_rnn.latest), "perf_mat"
)
swp_grids = s_minus_u_grids(
    animal_and_rnn(gd.abstract_swp_probability_matrix_rnn.latest), "swp_mat"
)

# flip response: P(Switch | flipped) - P(Switch | same), from the previous block's
# last 10 trials into the new block's first 50
flip_df = animal_and_rnn(gd.abstract_swp_around_flip_rnn.latest)
wide = flip_df.pivot_table(
    index=["kind", "group", "name", "trial_id"],
    columns="prev_best_arm",
    values="switch_prob",
)
resp_df = (wide["flipped"] - wide["same"]).rename("flip_resp").reset_index()

# each subject's peak flip response after the transition (gaussian-smoothed), as in
# the Cosyne abstract: its trial and height
sigma = 2  # smoothing (trials)
rows = []
for (kind, group, name), d in resp_df[resp_df["trial_id"] > 0].groupby(
    ["kind", "group", "name"]
):
    smooth = gaussian_filter1d(np.nan_to_num(d["flip_resp"].to_numpy()), sigma)
    rows.append(
        dict(
            kind=kinds[kind],
            group=group,
            name=name,
            peak_trial=d["trial_id"].to_numpy()[np.argmax(smooth)],
            peak_swp=smooth.max(),
        )
    )
peaks = pd.DataFrame(rows)

fig = plotting.Fig(2, 4, size=(17, 9), fontsize=14, axis_lw=1.5, tick_size=5)

# ---- top row: struc - unstruc prob grids, animals vs RNNs ----
for col, (name, (probs, diffs, counts), mat_col, label) in enumerate(
    [
        ("P(High)", perf_grids, "perf_mat", "Diff. P(High)"),
        ("P(Switch)", swp_grids, "swp_mat", "Diff. P(Switch)"),
    ]
):
    # shared color scale so the animal and RNN struc advantage are comparable
    colors = diverging_bins(
        np.nanmax([np.nanmax(np.abs(d)) for d in diffs.values()]),
        step=BIN_STEPS[mat_col],
    )
    axes = []
    for i, (k, kind_label) in enumerate(kinds.items()):
        ax = fig.subplot(fig.gs[0, 2 * col + i])
        im = draw_prob_grid(
            ax,
            diffs[k],
            probs,
            has_diag=np.isfinite(np.diag(diffs[k])).any(),  # P(High) has none
            **colors,
        )
        ax.set_title(
            f"{name}: {kind_label}\nstruc − unstruc "
            f"(n={counts[f'struc_{k}']} vs {counts[f'unstruc_{k}']})"
        )
        if i > 0:
            ax.set_ylabel("")
        axes.append(ax)
    fig.fig.colorbar(im, ax=axes, label=label, shrink=0.8)

# ---- bottom row: P(Switch) around the transition (rows: animals, RNNs; columns:
# raw flipped, raw same, flipped - same), then peak height and peak trial ----
curve_gs = fig.gs[1, 0:2].subgridspec(2, 3)
curve_cols = [
    ("flipped", flip_df[flip_df["prev_best_arm"] == "flipped"], "switch_prob"),
    ("same", flip_df[flip_df["prev_best_arm"] == "same"], "switch_prob"),
    ("flipped − same", resp_df, "flip_resp"),
]
for r, (k, kind_label) in enumerate(kinds.items()):
    row_axes = []
    for j, (title, data, y) in enumerate(curve_cols):
        ax = fig.subplot(curve_gs[r, j])
        sns.lineplot(
            data=data[data["kind"] == k],
            x="trial_id",
            y=y,
            hue="group",
            hue_order=hue_order,
            palette=palette,
            lw=2,
            err_kws=dict(edgecolor="none", alpha=0.3),
            errorbar="se",
            legend=False,
            ax=ax,
        )
        ax.axvline(0, color="gray", ls=":", lw=1.2)  # block transition
        ax.grid(axis="y", zorder=-1, alpha=0.5)
        ax.set_xlim(-10, 50)
        ax.set_xticks([-10, 20, 50])  # transition marked by the dotted line
        ax.set_xlabel("Trials from transition" if r == 1 else "")
        ax.set_ylabel(f"{kind_label}\nP(Switch)" if j == 0 else "")
        if r == 0:
            ax.set_title(title)
        row_axes.append(ax)

    # raw flipped and same share a y-scale; ticks at 0, middle, top
    hi = max(ax.dataLim.y1 for ax in row_axes[:2])
    for ax in row_axes[:2]:
        three_yticks(ax, lo=0.0, hi=hi)
    ax = row_axes[2]
    ax.axhline(0, color="gray", lw=1)
    top = three_yticks(ax, lo=0.0)[-1]
    ax.set_ylim(-0.1 * top, top)  # room for small dips below 0


def plot_peak_box(ax, data, y, ylabel, set_yaxis):
    plotter = (
        SeabornPlotter(
            data=data,
            x="kind",
            y=y,
            hue="group",
            hue_order=hue_order,
            ax=ax,
        )
        .boxplot_filled(palette=palette, sep=0.4)
        .stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=5,
            jitter=0.03,
            offset=0.1,  # dots beside, not on, their box
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
    )
    set_yaxis(ax)  # before the brackets (data units)
    plotter.stat_classical(
        "Mann-Whitney",
        pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
        loc="outside",
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)


# peak height: RNNs switch ~10x more than animals, so each kind gets its own y-axis
height_gs = fig.gs[1, 2].subgridspec(1, 2)
for i, kind_label in enumerate(kinds.values()):
    plot_peak_box(
        fig.subplot(height_gs[i]),
        peaks[peaks["kind"] == kind_label],
        "peak_swp",
        "Flip response\npeak Diff. P(Switch)" if i == 0 else "",
        lambda ax: three_yticks(ax, lo=0.0),
    )

# peak trial: same trial scale for animals and RNNs
plot_peak_box(
    fig.subplot(fig.gs[1, 3]),
    peaks,
    "peak_trial",
    "Flip response\npeak (trial)",
    lambda ax: (ax.set_ylim(0, 50), ax.set_yticks([0, 25, 50])),
)

print(peaks.groupby(["kind", "group"])[["peak_trial", "peak_swp"]].median())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "animal_rnn_8020", format="png", dpi=300)
fig.savefig(figdir / "animal_rnn_8020", format="svg")